# Faz 2.1 (v2) — Sağlam LM Benchmark (TinyShakespeare)

Önceki testteki sorunlar düzeltildi:
- Her model **2 farklı LR** ile test edilir, en iyi LR seçilir.
- Her konfigürasyon **3 farklı seed** ile tekrarlanır.
- Early stopping patience **7**'ye çıkarıldı.
- max_iters **5000**'e çıkarıldı.
- **2× T4 GPU paralel kullanım** — koşular 2 GPU'ya dağıtılır, süre ~yarıya düşer.

Toplam koşu: 2 model × 2 LR × 3 seed = **12 koşu** (6 koşu/GPU, paralel)

In [ ]:
# --- KURULUM ---
import subprocess, sys, os, urllib.request
REPO = "/kaggle/working/HFP"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/kayra-hn/HFP.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.40"], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
os.environ["HFP_CKPT_DIR"] = "/kaggle/working/ck"
os.environ["PYTHONPATH"] = REPO

data_path = 'tinyshakespeare.txt'
if not os.path.exists(data_path):
    print("Downloading tinyshakespeare...")
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    urllib.request.urlretrieve(url, data_path)

import torch
num_gpus = torch.cuda.device_count()
print(f"Bulunan GPU sayisi: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
if num_gpus < 2:
    print("UYARI: 2 GPU bulunamadi, kosular sirayla tek GPU'da calisacak.")
print("Kurulum tamam!")

In [ ]:
# --- 12 KOSU: 2 model x 2 LR x 3 seed --- 2x T4 PARALEL ---
import subprocess, sys, os, time
from concurrent.futures import ThreadPoolExecutor, as_completed
import torch

models = ["gpt2", "hfp"]
lrs = [5e-4, 1e-3]
seeds = [0, 1, 2]

# Tum kosu konfigurasyonlarini olustur
all_runs = []
for model in models:
    for lr in lrs:
        for seed in seeds:
            tag = f"{model}_lr{lr}_s{seed}"
            all_runs.append((model, lr, seed, tag))

print(f"Toplam {len(all_runs)} kosu planlanadi.")

num_gpus = torch.cuda.device_count()
print(f"Kullanilacak GPU sayisi: {num_gpus}")

def run_single(model, lr, seed, tag, gpu_id):
    """Tek bir kosuyu belirtilen GPU'da calistir."""
    env = os.environ.copy()
    env["PYTHONHASHSEED"] = str(seed)
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    
    cmd = [
        sys.executable, "train.py",
        "--model", model,
        "--learning_rate", str(lr),
        "--max_iters", "5000",
        "--eval_interval", "200",
        "--patience", "7",
        "--batch_size", "16",
        "--seq_length", "256",
        "--warmup_steps", "100",
        "--seed", str(seed),
        "--log_tag", tag
    ]
    
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    elapsed = time.time() - t0
    
    output_tail = result.stdout[-1500:] if len(result.stdout) > 1500 else result.stdout
    status = "OK" if result.returncode == 0 else "HATA"
    error_msg = ""
    if result.returncode != 0:
        error_msg = result.stderr[-500:]
    
    return tag, gpu_id, elapsed, status, output_tail, error_msg

# --- PARALEL CALISTIRMA ---
if num_gpus >= 2:
    # 2 GPU: round-robin dagitim, 2 worker ile paralel
    gpu_assignments = [(run, i % num_gpus) for i, run in enumerate(all_runs)]
    
    print(f"\n{'='*60}")
    print(f"  2x GPU PARALEL MOD - Her GPU'da 6 kosu")
    print(f"{'='*60}")
    for (model, lr, seed, tag), gpu_id in gpu_assignments:
        print(f"  GPU {gpu_id}: {tag}")
    print(f"{'='*60}\n")
    
    total_start = time.time()
    completed = 0
    
    with ThreadPoolExecutor(max_workers=num_gpus) as executor:
        futures = {}
        for (model, lr, seed, tag), gpu_id in gpu_assignments:
            future = executor.submit(run_single, model, lr, seed, tag, gpu_id)
            futures[future] = tag
        
        for future in as_completed(futures):
            tag, gpu_id, elapsed, status, output_tail, error_msg = future.result()
            completed += 1
            print(f"\n{'='*60}")
            print(f"  [{completed}/{len(all_runs)}] {tag} (GPU {gpu_id}) - {status} - {elapsed:.1f}s")
            print(f"{'='*60}")
            print(output_tail)
            if error_msg:
                print(f"HATA: {error_msg}")
    
    total_elapsed = time.time() - total_start
    print(f"\n{'='*60}")
    print(f"  TUM KOSULAR TAMAMLANDI - Toplam sure: {total_elapsed:.1f}s ({total_elapsed/60:.1f} dk)")
    print(f"  (Paralel kazanim: 2 GPU ile ~2x hiz)")
    print(f"{'='*60}")

else:
    # Tek GPU fallback: sirayla calistir
    print(f"\n{'='*60}")
    print(f"  TEK GPU MODU - 12 kosu sirayla")
    print(f"{'='*60}\n")
    
    total_start = time.time()
    for i, (model, lr, seed, tag) in enumerate(all_runs):
        tag_result, _, elapsed, status, output_tail, error_msg = run_single(model, lr, seed, tag, 0)
        print(f"\n{'='*60}")
        print(f"  [{i+1}/{len(all_runs)}] {tag_result} - {status} - {elapsed:.1f}s")
        print(f"{'='*60}")
        print(output_tail)
        if error_msg:
            print(f"HATA: {error_msg}")
    
    total_elapsed = time.time() - total_start
    print(f"\n{'='*60}")
    print(f"  TUM KOSULAR TAMAMLANDI - Toplam sure: {total_elapsed:.1f}s ({total_elapsed/60:.1f} dk)")
    print(f"{'='*60}")

In [ ]:
# --- SONUCLARI TOPLA VE TABLOLASTIR ---
import pandas as pd
import glob, csv, os

models = ["gpt2", "hfp"]
lrs = [5e-4, 1e-3]
seeds = [0, 1, 2]

all_results = []

for model in models:
    for lr in lrs:
        for seed in seeds:
            tag = f"{model}_lr{lr}_s{seed}"
            log_file = f"{tag}_log.csv"
            if os.path.exists(log_file):
                df = pd.read_csv(log_file)
                best_val = df['val_loss'].min()
                best_ppl = df.loc[df['val_loss'].idxmin(), 'val_perplexity']
                best_step = df.loc[df['val_loss'].idxmin(), 'step']
                final_train = df.iloc[-1]['train_loss']
                all_results.append({
                    'Model': model.upper(),
                    'LR': lr,
                    'Seed': seed,
                    'Best Val Loss': round(best_val, 4),
                    'Best PPL': round(best_ppl, 2),
                    'Best Step': int(best_step),
                    'Final Train Loss': round(final_train, 4),
                    'Overfit Gap': round(final_train - best_val, 4)
                })
            else:
                print(f"UYARI: {log_file} bulunamadi!")

df_all = pd.DataFrame(all_results)
print("\n" + "="*70)
print("  TUM KOSULARIN DETAYLI SONUCLARI")
print("="*70)
print(df_all.to_string(index=False))

# Her model icin en iyi LR'yi sec
print("\n" + "="*70)
print("  MODEL BAZINDA EN IYI LR SECIMI (3-seed ortalamasi)")
print("="*70)
summary = df_all.groupby(['Model', 'LR']).agg(
    mean_val_loss=('Best Val Loss', 'mean'),
    std_val_loss=('Best Val Loss', 'std'),
    mean_ppl=('Best PPL', 'mean'),
    std_ppl=('Best PPL', 'std'),
    mean_overfit=('Overfit Gap', 'mean')
).round(4)
print(summary)

# Nihai adil karsilastirma
print("\n" + "="*70)
print("  NIHAI ADIL KARSILASTIRMA")
print("="*70)
for model in ['GPT2', 'HFP']:
    model_data = df_all[df_all['Model'] == model]
    best_lr_row = summary.loc[model].sort_values('mean_val_loss').iloc[0]
    best_lr = summary.loc[model].sort_values('mean_val_loss').index[0]
    best_lr_data = model_data[model_data['LR'] == best_lr]
    mean_loss = best_lr_data['Best Val Loss'].mean()
    std_loss = best_lr_data['Best Val Loss'].std()
    mean_ppl = best_lr_data['Best PPL'].mean()
    std_ppl = best_lr_data['Best PPL'].std()
    print(f"{model} (best LR={best_lr}): Val Loss = {mean_loss:.4f} +/- {std_loss:.4f}  |  PPL = {mean_ppl:.2f} +/- {std_ppl:.2f}")